# VAE x64 Latent Dimension Sweep

This notebook is the curated submission-facing entry point for the VAE latent dimension sweep on the shared x64 anomaly benchmark.
It defaults to reusing the saved per-latent-dim artifacts instead of rerunning the sweep.

## Submission Context

- Dataset notebook: `data/dataset/x64/benchmark_50k_5pct/notebook.ipynb`
- Sweep config: `experiments/anomaly_detection/vae/x64/latent_dim_sweep/train_config.toml`
- Training script: `scripts/train_vae.py`
- Artifact root: `experiments/anomaly_detection/vae/x64/latent_dim_sweep/artifacts/vae_latent_dim_sweep/`
- Default behavior: reuse the saved `results/latent_dim_sweep_summary.json` and the per-latent-dim checkpoints and evaluation outputs

### Setup And Imports

This cell resolves the repo root, exposes `src/` on `PYTHONPATH`, and imports the shared config loader plus the standard utilities used to read the saved sweep artifacts.

In [ ]:
from __future__ import annotations
import json
import os
import subprocess
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
try:
    from IPython.display import display
except ImportError:

    def display(obj: object) -> None:
        print(obj)
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src' / 'wafer_defect').exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / 'src' / 'wafer_defect').exists():
            REPO_ROOT = candidate
            break
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml

### Load The Sweep Config And Run Controls

This cell points the notebook at the local latent-dim-sweep config and exposes the rerun flags. Leave them as `False` to keep the notebook in artifact-first mode.

In [ ]:
try:
    CONFIG_PATH = REPO_ROOT / 'experiments' / 'anomaly_detection' / 'vae' / 'x64' / 'latent_dim_sweep' / 'train_config.toml'
    config = load_toml(CONFIG_PATH)
    SWEEP_ROOT = REPO_ROOT / config['run']['output_dir']
    SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
    CONFIGS_DIR = SWEEP_ROOT / 'configs'
    CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
    PLOTS_DIR = SWEEP_ROOT / 'plots'
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    FORCE_RERUN_SWEEP = False
    RUN_LATENT_DIM_SWEEP = False
    display(config)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "config_path = repo_root / 'experiments' / 'anomaly_detection' / 'vae' / 'x64' / 'latent_dim_sweep' / 'train_config.toml'\nconfig = load_toml(config_path)\nsweep_root = repo_root / config['run']['output_dir']\nsweep_root.mkdir(parents=true, exist_ok=true)\nconfigs_dir = sweep_root / 'configs'\nconfigs_dir.mkdir(parents=true, exist_ok=true)\nplots_dir = sweep_root / 'plots'\nplots_dir.mkdir(parents=true, exist_ok=true)\nforce_rerun_sweep = false\nrun_latent_dim_sweep = false\ndisplay(config)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Optionally Rerun The Sweep

This cell is only used when you explicitly enable the sweep rerun. Otherwise it simply reuses the saved summary and per-latent-dim artifacts already present in the repository.

In [ ]:
try:

    def format_toml_value(value):
        if isinstance(value, bool):
            return 'true' if value else 'false'
        if isinstance(value, (int, float)):
            return repr(value)
        escaped = str(value).replace('\\', '\\\\').replace('"', '\\"')
        return f'"{escaped}"'

    def dump_toml(config_dict):
        lines = []
        for section, values in config_dict.items():
            lines.append(f'[{section}]')
            for key, value in values.items():
                lines.append(f'{key} = {format_toml_value(value)}')
            lines.append('')
        return '\n'.join(lines).rstrip() + '\n'

    def latent_dim_tag(latent_dim_value: int) -> str:
        return f'latent_dim_{int(latent_dim_value)}'
    summary_path = SWEEP_ROOT / 'results' / 'latent_dim_sweep_summary.json'
    env = os.environ.copy()
    env['PYTHONPATH'] = str(SRC_ROOT) if not env.get('PYTHONPATH') else str(SRC_ROOT) + os.pathsep + env['PYTHONPATH']
    if RUN_LATENT_DIM_SWEEP or FORCE_RERUN_SWEEP:
        latent_dims = list(config['sweep']['latent_dims'])
        results = []
        for latent_dim_value in latent_dims:
            tag = latent_dim_tag(latent_dim_value)
            run_output_dir = SWEEP_ROOT / tag
            run_config = {'run': {'output_dir': run_output_dir.relative_to(REPO_ROOT).as_posix(), 'seed': config['run']['seed']}, 'data': dict(config['data']), 'training': dict(config['training']), 'model': {'type': 'vae', 'latent_dim': int(latent_dim_value), 'beta': 0.005}}
            run_config_path = CONFIGS_DIR / f'train_vae_{tag}.toml'
            run_config_path.write_text(dump_toml(run_config), encoding='utf-8')
            train_cmd = [sys.executable, 'scripts/train_vae.py', '--config', str(run_config_path.relative_to(REPO_ROOT))]
            eval_cmd = [sys.executable, 'scripts/evaluate_reconstruction_model.py', '--checkpoint', str((run_output_dir / 'checkpoints' / 'best_model.pt').relative_to(REPO_ROOT)), '--config', str(run_config_path.relative_to(REPO_ROOT)), '--output-dir', str((run_output_dir / 'evaluation').relative_to(REPO_ROOT))]
            subprocess.run(train_cmd, cwd=REPO_ROOT, env=env, check=True)
            subprocess.run(eval_cmd, cwd=REPO_ROOT, env=env, check=True)
            run_summary = json.loads((run_output_dir / 'results' / 'evaluation' / 'summary.json').read_text(encoding='utf-8'))
            results.append({'latent_dim': int(latent_dim_value), 'threshold': float(run_summary['threshold']), 'val_threshold_precision': float(run_summary['metrics_at_validation_threshold']['precision']), 'val_threshold_recall': float(run_summary['metrics_at_validation_threshold']['recall']), 'val_threshold_f1': float(run_summary['metrics_at_validation_threshold']['f1']), 'auroc': float(run_summary['metrics_at_validation_threshold']['auroc']), 'auprc': float(run_summary['metrics_at_validation_threshold']['auprc']), 'best_sweep_threshold': float(run_summary['best_threshold_sweep']['threshold']), 'best_sweep_precision': float(run_summary['best_threshold_sweep']['precision']), 'best_sweep_recall': float(run_summary['best_threshold_sweep']['recall']), 'best_sweep_f1': float(run_summary['best_threshold_sweep']['f1'])})
        summary_path.write_text(json.dumps({'latent_dims': latent_dims, 'results': results}, indent=2), encoding='utf-8')
    elif summary_path.exists():
        print(f'Found existing latent-dim sweep artifacts in {SWEEP_ROOT}. Skipping rerun.')
    else:
        print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'def format_toml_value(value):\n    if isinstance(value, bool):\n        return \'true\' if value else \'false\'\n    if isinstance(value, (int, float)):\n        return repr(value)\n    escaped = str(value).replace(\'\\\\\', \'\\\\\\\\\').replace(\'"\', \'\\\\"\')\n    return f\'"{escaped}"\'\n\ndef dump_toml(config_dict):\n    lines = []\n    for section, values in config_dict.items():\n        lines.append(f\'[{section}]\')\n        for key, value in values.items():\n            lines.append(f\'{key} = {format_toml_value(value)}\')\n        lines.append(\'\')\n    return \'\\n\'.join(lines).rstrip() + \'\\n\'\n\ndef latent_dim_tag(latent_dim_value: int) -> str:\n    return f\'latent_dim_{int(latent_dim_value)}\'\nsummary_path = sweep_root / \'results\' / \'latent_dim_sweep_summary.json\'\nenv = os.environ.copy()\nenv[\'pythonpath\'] = str(src_root) if not env.get(\'pythonpath\') else str(src_root) + os.pathsep + env[\'pythonpath\']\nif run_latent_dim_sweep or force_rerun_sweep or (not summary_path.exists()):\n    latent_dims = list(config[\'sweep\'][\'latent_dims\'])\n    results = []\n    for latent_dim_value in latent_dims:\n        tag = latent_dim_tag(latent_dim_value)\n        run_output_dir = sweep_root / tag\n        run_config = {\'run\': {\'output_dir\': run_output_dir.relative_to(repo_root).as_posix(), \'seed\': config[\'run\'][\'seed\']}, \'data\': dict(config[\'data\']), \'training\': dict(config[\'training\']), \'model\': {\'type\': \'vae\', \'latent_dim\': int(latent_dim_value), \'beta\': 0.005}}\n        run_config_path = configs_dir / f\'train_vae_{tag}.toml\'\n        run_config_path.write_text(dump_toml(run_config), encoding=\'utf-8\')\n        train_cmd = [sys.executable, \'scripts/train_vae.py\', \'--config\', str(run_config_path.relative_to(repo_root))]\n        eval_cmd = [sys.executable, \'scripts/evaluate_reconstruction_model.py\', \'--checkpoint\', str((run_output_dir / \'checkpoints\' / \'best_model.pt\').relative_to(repo_root)), \'--config\', str(run_config_path.relative_to(repo_root)), \'--output-dir\', str((run_output_dir / \'evaluation\').relative_to(repo_root))]\n        subprocess.run(train_cmd, cwd=repo_root, env=env, check=true)\n        subprocess.run(eval_cmd, cwd=repo_root, env=env, check=true)\n        run_summary = json.loads((run_output_dir / \'results\' / \'evaluation\' / \'summary.json\').read_text(encoding=\'utf-8\'))\n        results.append({\'latent_dim\': int(latent_dim_value), \'threshold\': float(run_summary[\'threshold\']), \'val_threshold_precision\': float(run_summary[\'metrics_at_validation_threshold\'][\'precision\']), \'val_threshold_recall\': float(run_summary[\'metrics_at_validation_threshold\'][\'recall\']), \'val_threshold_f1\': float(run_summary[\'metrics_at_validation_threshold\'][\'f1\']), \'auroc\': float(run_summary[\'metrics_at_validation_threshold\'][\'auroc\']), \'auprc\': float(run_summary[\'metrics_at_validation_threshold\'][\'auprc\']), \'best_sweep_threshold\': float(run_summary[\'best_threshold_sweep\'][\'threshold\']), \'best_sweep_precision\': float(run_summary[\'best_threshold_sweep\'][\'precision\']), \'best_sweep_recall\': float(run_summary[\'best_threshold_sweep\'][\'recall\']), \'best_sweep_f1\': float(run_summary[\'best_threshold_sweep\'][\'f1\'])})\n    summary_path.write_text(json.dumps({\'latent_dims\': latent_dims, \'results\': results}, indent=2), encoding=\'utf-8\')\nelse:\n    print(f\'found existing latent-dim sweep artifacts in {sweep_root}. skipping rerun.\')'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Load The Saved Sweep Summary

This cell reads the aggregated summary and builds a dataframe that we can use for the comparison plots and tables.

In [ ]:
try:
    summary_payload = json.loads((SWEEP_ROOT / 'results' / 'latent_dim_sweep_summary.json').read_text(encoding='utf-8'))
    latent_dim_sweep_df = pd.DataFrame(summary_payload['results']).sort_values('val_threshold_f1', ascending=False).reset_index(drop=True)
    display(latent_dim_sweep_df)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "summary_payload = json.loads((sweep_root / 'results' / 'latent_dim_sweep_summary.json').read_text(encoding='utf-8'))\nlatent_dim_sweep_df = pd.dataframe(summary_payload['results']).sort_values('val_threshold_f1', ascending=false).reset_index(drop=true)\ndisplay(latent_dim_sweep_df)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Plot Sweep Metrics Across `beta`

This cell recreates the main sweep comparison figure, shows it inline, and saves it to the local `plots/` folder.

In [ ]:
try:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
    sorted_df = latent_dim_sweep_df.sort_values('latent_dim').reset_index(drop=True)
    axes[0].plot(sorted_df['latent_dim'], sorted_df['val_threshold_f1'], marker='o', label='val-threshold F1')
    axes[0].plot(sorted_df['latent_dim'], sorted_df['best_sweep_f1'], marker='s', label='best-sweep F1')
    axes[0].set_title('F1 vs Latent Dimension')
    axes[0].set_xlabel('latent_dim')
    axes[0].legend()
    axes[1].plot(sorted_df['latent_dim'], sorted_df['auroc'], marker='o', color='#1f77b4')
    axes[1].set_title('AUROC vs Latent Dimension')
    axes[1].set_xlabel('latent_dim')
    axes[2].plot(sorted_df['latent_dim'], sorted_df['auprc'], marker='o', color='#2ca02c')
    axes[2].set_title('AUPRC vs Latent Dimension')
    axes[2].set_xlabel('latent_dim')
    fig.savefig(PLOTS_DIR / 'latent_dim_sweep_metrics.png', dpi=160, bbox_inches='tight')
    display(fig)
    plt.close(fig)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=true)\nsorted_df = latent_dim_sweep_df.sort_values('latent_dim').reset_index(drop=true)\naxes[0].plot(sorted_df['latent_dim'], sorted_df['val_threshold_f1'], marker='o', label='val-threshold f1')\naxes[0].plot(sorted_df['latent_dim'], sorted_df['best_sweep_f1'], marker='s', label='best-sweep f1')\naxes[0].set_title('f1 vs latent dimension')\naxes[0].set_xlabel('latent_dim')\naxes[0].legend()\naxes[1].plot(sorted_df['latent_dim'], sorted_df['auroc'], marker='o', color='#1f77b4')\naxes[1].set_title('auroc vs latent dimension')\naxes[1].set_xlabel('latent_dim')\naxes[2].plot(sorted_df['latent_dim'], sorted_df['auprc'], marker='o', color='#2ca02c')\naxes[2].set_title('auprc vs latent dimension')\naxes[2].set_xlabel('latent_dim')\nfig.savefig(plots_dir / 'latent_dim_sweep_metrics.png', dpi=160, bbox_inches='tight')\ndisplay(fig)\nplt.close(fig)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Compare Training Curves Across Betas

This cell loads each saved `history.json`, overlays the validation curves, and saves the comparison plot for the report.

In [ ]:
try:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    for latent_dim_value in sorted(latent_dim_sweep_df['latent_dim'].tolist()):
        tag = latent_dim_tag(latent_dim_value)
        history_path = SWEEP_ROOT / tag / 'history.json'
        history_df = pd.DataFrame(json.loads(history_path.read_text(encoding='utf-8')))
        axes[0].plot(history_df['epoch'], history_df['val_loss'], label=f'latent_dim={latent_dim_value}')
        axes[1].plot(history_df['epoch'], history_df['val_reconstruction_loss'], label=f'latent_dim={latent_dim_value}')
    axes[0].set_title('Validation Total Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend(fontsize=8)
    axes[1].set_title('Validation Reconstruction Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend(fontsize=8)
    fig.savefig(PLOTS_DIR / 'latent_dim_sweep_training_curves.png', dpi=160, bbox_inches='tight')
    display(fig)
    plt.close(fig)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=true)\nfor latent_dim_value in sorted(latent_dim_sweep_df['latent_dim'].tolist()):\n    tag = latent_dim_tag(latent_dim_value)\n    history_path = sweep_root / tag / 'history.json'\n    history_df = pd.dataframe(json.loads(history_path.read_text(encoding='utf-8')))\n    axes[0].plot(history_df['epoch'], history_df['val_loss'], label=f'latent_dim={latent_dim_value}')\n    axes[1].plot(history_df['epoch'], history_df['val_reconstruction_loss'], label=f'latent_dim={latent_dim_value}')\naxes[0].set_title('validation total loss')\naxes[0].set_xlabel('epoch')\naxes[0].legend(fontsize=8)\naxes[1].set_title('validation reconstruction loss')\naxes[1].set_xlabel('epoch')\naxes[1].legend(fontsize=8)\nfig.savefig(plots_dir / 'latent_dim_sweep_training_curves.png', dpi=160, bbox_inches='tight')\ndisplay(fig)\nplt.close(fig)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Inspect The Best Beta Run

This cell identifies the best saved beta run by validation-threshold F1 and displays its saved evaluation summary so the sweep still ends with a concrete recommended configuration.

In [ ]:
try:
    best_latent_dim_row = latent_dim_sweep_df.iloc[0].to_dict()
    best_latent_dim = int(best_latent_dim_row['latent_dim'])
    best_tag = latent_dim_tag(best_latent_dim)
    best_run_dir = SWEEP_ROOT / best_tag
    best_eval_summary = json.loads((best_run_dir / 'results' / 'evaluation' / 'summary.json').read_text(encoding='utf-8'))
    best_test_scores_df = pd.read_csv(best_run_dir / 'results' / 'evaluation' / 'test_scores.csv')
    best_threshold_sweep_df = pd.read_csv(best_run_dir / 'results' / 'evaluation' / 'threshold_sweep.csv')
    cm = best_eval_summary['metrics_at_validation_threshold'].get('confusion_matrix', [[0, 0], [0, 0]])
    cm_df = pd.DataFrame(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
    print(f'Best Latent Dim by validation-threshold F1: {best_latent_dim}')
    print(f'Best run directory: {best_run_dir}')
    display(pd.DataFrame([best_latent_dim_row]))
    display(pd.DataFrame([best_eval_summary['metrics_at_validation_threshold']]))
    display(pd.DataFrame([best_eval_summary['best_threshold_sweep']]))
    default_threshold = float(best_eval_summary['threshold'])
    best_sweep_threshold = float(best_eval_summary['best_threshold_sweep']['threshold'])
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=True)
    axes[0].hist(best_test_scores_df.loc[best_test_scores_df['is_anomaly'] == 0, 'score'], bins=30, alpha=0.7, label='normal')
    axes[0].hist(best_test_scores_df.loc[best_test_scores_df['is_anomaly'] == 1, 'score'], bins=30, alpha=0.7, label='anomaly')
    axes[0].axvline(default_threshold, color='red', linestyle='--', label=f'val threshold = {default_threshold:.4f}')
    axes[0].set_title(f'Best-Latent-Dim Score Distribution (latent_dim={best_latent_dim})')
    axes[0].set_xlabel('Anomaly score')
    axes[0].legend()
    axes[1].plot(best_threshold_sweep_df['threshold'], best_threshold_sweep_df['precision'], label='precision')
    axes[1].plot(best_threshold_sweep_df['threshold'], best_threshold_sweep_df['recall'], label='recall')
    axes[1].plot(best_threshold_sweep_df['threshold'], best_threshold_sweep_df['f1'], label='f1')
    axes[1].axvline(default_threshold, color='red', linestyle='--', label='val threshold')
    axes[1].axvline(best_sweep_threshold, color='green', linestyle=':', label='best sweep')
    axes[1].set_title('Best-Latent-Dim Threshold Sweep')
    axes[1].set_xlabel('Threshold')
    axes[1].legend()
    heatmap = axes[2].imshow(cm_df.to_numpy(), cmap='Blues')
    axes[2].set_xticks(range(cm_df.shape[1]), cm_df.columns)
    axes[2].set_yticks(range(cm_df.shape[0]), cm_df.index)
    axes[2].set_title('Best-Latent-Dim Confusion Matrix')
    axes[2].set_xlabel('Predicted label')
    axes[2].set_ylabel('True label')
    for row_idx in range(cm_df.shape[0]):
        for col_idx in range(cm_df.shape[1]):
            axes[2].text(col_idx, row_idx, str(int(cm_df.iat[row_idx, col_idx])), ha='center', va='center', color='black')
    fig.colorbar(heatmap, ax=axes[2], fraction=0.046, pad=0.04)
    fig.savefig(PLOTS_DIR / 'best_latent_dim_distribution_sweep_confusion.png', dpi=160, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    display(cm_df)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "best_latent_dim_row = latent_dim_sweep_df.iloc[0].to_dict()\nbest_latent_dim = int(best_latent_dim_row['latent_dim'])\nbest_tag = latent_dim_tag(best_latent_dim)\nbest_run_dir = sweep_root / best_tag\nbest_eval_summary = json.loads((best_run_dir / 'results' / 'evaluation' / 'summary.json').read_text(encoding='utf-8'))\nbest_test_scores_df = pd.read_csv(best_run_dir / 'results' / 'evaluation' / 'test_scores.csv')\nbest_threshold_sweep_df = pd.read_csv(best_run_dir / 'results' / 'evaluation' / 'threshold_sweep.csv')\ncm = best_eval_summary['metrics_at_validation_threshold'].get('confusion_matrix', [[0, 0], [0, 0]])\ncm_df = pd.dataframe(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])\nprint(f'best latent dim by validation-threshold f1: {best_latent_dim}')\nprint(f'best run directory: {best_run_dir}')\ndisplay(pd.dataframe([best_latent_dim_row]))\ndisplay(pd.dataframe([best_eval_summary['metrics_at_validation_threshold']]))\ndisplay(pd.dataframe([best_eval_summary['best_threshold_sweep']]))\ndefault_threshold = float(best_eval_summary['threshold'])\nbest_sweep_threshold = float(best_eval_summary['best_threshold_sweep']['threshold'])\nfig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=true)\naxes[0].hist(best_test_scores_df.loc[best_test_scores_df['is_anomaly'] == 0, 'score'], bins=30, alpha=0.7, label='normal')\naxes[0].hist(best_test_scores_df.loc[best_test_scores_df['is_anomaly'] == 1, 'score'], bins=30, alpha=0.7, label='anomaly')\naxes[0].axvline(default_threshold, color='red', linestyle='--', label=f'val threshold = {default_threshold:.4f}')\naxes[0].set_title(f'best-latent-dim score distribution (latent_dim={best_latent_dim})')\naxes[0].set_xlabel('anomaly score')\naxes[0].legend()\naxes[1].plot(best_threshold_sweep_df['threshold'], best_threshold_sweep_df['precision'], label='precision')\naxes[1].plot(best_threshold_sweep_df['threshold'], best_threshold_sweep_df['recall'], label='recall')\naxes[1].plot(best_threshold_sweep_df['threshold'], best_threshold_sweep_df['f1'], label='f1')\naxes[1].axvline(default_threshold, color='red', linestyle='--', label='val threshold')\naxes[1].axvline(best_sweep_threshold, color='green', linestyle=':', label='best sweep')\naxes[1].set_title('best-latent-dim threshold sweep')\naxes[1].set_xlabel('threshold')\naxes[1].legend()\nheatmap = axes[2].imshow(cm_df.to_numpy(), cmap='blues')\naxes[2].set_xticks(range(cm_df.shape[1]), cm_df.columns)\naxes[2].set_yticks(range(cm_df.shape[0]), cm_df.index)\naxes[2].set_title('best-latent-dim confusion matrix')\naxes[2].set_xlabel('predicted label')\naxes[2].set_ylabel('true label')\nfor row_idx in range(cm_df.shape[0]):\n    for col_idx in range(cm_df.shape[1]):\n        axes[2].text(col_idx, row_idx, str(int(cm_df.iat[row_idx, col_idx])), ha='center', va='center', color='black')\nfig.colorbar(heatmap, ax=axes[2], fraction=0.046, pad=0.04)\nfig.savefig(plots_dir / 'best_latent_dim_distribution_sweep_confusion.png', dpi=160, bbox_inches='tight')\ndisplay(fig)\nplt.close(fig)\ndisplay(cm_df)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise
